# Kaggle Full 02 - Training (Self-Contained)

This notebook is fully self-contained and trains a baseline liveness model from manifests.


In [ ]:
# !pip install -q torch torchvision opencv-python pandas numpy tqdm

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [ ]:
@dataclass
class TrainConfig:
    train_manifest: str
    val_manifest: str
    output_dir: str
    image_size: int = 80
    batch_size: int = 128
    epochs: int = 12
    lr: float = 1e-3
    num_workers: int = 2
    seed: int = 42


class ManifestDataset(Dataset):
    LEGACY_PREP_ROOT = Path('/kaggle/working/celeba_spoof_prepared_full')

    def __init__(self, manifest_path: str, image_size: int = 80):
        self.df = pd.read_csv(manifest_path)
        self.image_size = image_size
        self.manifest_path = Path(manifest_path)
        self.prepared_root = self.manifest_path.parent.parent

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path: str) -> Path:
        text = str(raw_path)
        path = Path(text)

        if path.exists():
            return path

        legacy_prefix = str(self.LEGACY_PREP_ROOT) + '/'
        if text.startswith(legacy_prefix):
            suffix = text[len(legacy_prefix):]
            candidate = self.prepared_root / suffix
            if candidate.exists():
                return candidate

        marker = 'crops_80x80/'
        if marker in text:
            suffix = text.split(marker, 1)[1]
            candidate = self.prepared_root / 'crops_80x80' / suffix
            if candidate.exists():
                return candidate

        if not path.is_absolute():
            candidate = self.prepared_root / path
            if candidate.exists():
                return candidate

        return path

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        resolved_path = self._resolve_image_path(row.image_path)
        image = cv2.imread(str(resolved_path))
        if image is None:
            raise RuntimeError(f'Could not load image: {row.image_path} (resolved: {resolved_path})')

        image = cv2.resize(image, (self.image_size, self.image_size))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        x = torch.from_numpy(image)
        y = torch.tensor(int(row.label), dtype=torch.long)
        return x, y


class SmallFASNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
PREP_ROOT = Path('/kaggle/input/datasets/doraemongwa/celeba-spoof-prepared-full/celeba_spoof_prepared_full/manifests')
OUT_ROOT = Path('/kaggle/working/celeba_spoof_training_full')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

cfg = TrainConfig(
    train_manifest=str(PREP_ROOT / 'train.csv'),
    val_manifest=str(PREP_ROOT / 'val.csv'),
    output_dir=str(OUT_ROOT),
)
print(cfg)

In [ ]:
torch.manual_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_ds = ManifestDataset(cfg.train_manifest, image_size=cfg.image_size)
val_ds = ManifestDataset(cfg.val_manifest, image_size=cfg.image_size)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

model = SmallFASNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

best_val_acc = -1.0
history = []
best_ckpt = Path(cfg.output_dir) / 'best_model.pt'


In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += float(loss.item()) * labels.size(0)
            total_correct += int((logits.argmax(dim=1) == labels).sum().item())
            total_count += int(labels.size(0))

    return {
        'loss': total_loss / max(total_count, 1),
        'acc': total_correct / max(total_count, 1),
    }


In [ ]:
for epoch in range(1, cfg.epochs + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.epochs}'):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += float(loss.item()) * labels.size(0)
        train_correct += int((logits.argmax(dim=1) == labels).sum().item())
        train_count += int(labels.size(0))

    val_metrics = evaluate(model, val_loader)

    row = {
        'epoch': epoch,
        'train_loss': train_loss / max(train_count, 1),
        'train_acc': train_correct / max(train_count, 1),
        'val_loss': val_metrics['loss'],
        'val_acc': val_metrics['acc'],
    }
    history.append(row)
    print(row)

    if val_metrics['acc'] > best_val_acc:
        best_val_acc = val_metrics['acc']
        torch.save({'state_dict': model.state_dict(), 'image_size': cfg.image_size}, best_ckpt)


In [ ]:
scripted_path = Path(cfg.output_dir) / 'best_model_scripted.pt'
model.eval()
scripted = torch.jit.script(model.cpu())
scripted.save(str(scripted_path))

history_path = Path(cfg.output_dir) / 'history.json'
history_path.write_text(json.dumps(history, indent=2))

summary = {
    'best_checkpoint': str(best_ckpt),
    'best_scripted_checkpoint': str(scripted_path),
    'best_val_acc': float(best_val_acc),
}
summary_path = Path(cfg.output_dir) / 'run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

print('best_val_acc:', best_val_acc)
print('best_ckpt:', best_ckpt)
print('scripted:', scripted_path)
print('history:', history_path)
print('summary:', summary_path)